In [52]:

import os
import cv2
import time
import threading
import numpy as np

from PIL import Image, ImageDraw, ImageFont
from concurrent.futures import ThreadPoolExecutor

# Keep OpenCV from creating additional threads
cv2.setNumThreads(1)

# CONFIGURATION
INPUT_DIR = "images_dataset"
OUTPUT_DIR = "sample_output/output_parallel"
EDGES_DIR = "sample_output/output_parallel_edges"

WATERMARK_TEXT = "ARTI 503 Parallel Project"
RESIZE_SHAPE = (128, 128)
FONT_PATH = None
BLUR_KERNEL = (5, 5)

# Number of workers
MAX_WORKERS = 8

# SHARED VARIABLES 
 
processed_counter = 0   
error_records = []  

# IMAGE PROCESSING FUNCTIONS


def add_watermark(image_pil, text, font_path=None):

    draw = ImageDraw.Draw(image_pil)

    if font_path and os.path.exists(font_path):
        font = ImageFont.truetype(font_path, 14)
    else:
        font = ImageFont.load_default()

    left, top, right, bottom = draw.textbbox(
        (0, 0), text, font=font
    )

    text_w = right - left
    text_h = bottom - top

    x = image_pil.width - text_w - 8
    y = image_pil.height - text_h - 8

    draw.text((x, y), text, fill=(255, 255, 255), font=font)

    return image_pil


def detect_edges(img_bgr):

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    blurred = cv2.GaussianBlur(
        gray, BLUR_KERNEL, 0
    )

    grad_x = cv2.Sobel(
        blurred, cv2.CV_64F, 1, 0, ksize=3
    )

    grad_y = cv2.Sobel(
        blurred, cv2.CV_64F, 0, 1, ksize=3
    )

    magnitude = cv2.magnitude(grad_x, grad_y)

    return cv2.convertScaleAbs(magnitude)


# PROCESS ONE IMAGE

def process_image(task):

    global processed_counter,error_records

    img_path, output_path, edges_path = task

    try:

        img = cv2.imread(img_path)

        if img is None:
            raise ValueError("Invalid image")

        # Sobel edge detection
        edges = detect_edges(img)

        cv2.imwrite(edges_path, edges)

        # Resize
        img_resized = cv2.resize(img, RESIZE_SHAPE)

        # Add watermark
        img_pil = Image.fromarray(
            cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        )

        img_pil = add_watermark(
            img_pil, WATERMARK_TEXT, FONT_PATH
        )

        final_img = cv2.cvtColor(
            np.array(img_pil),
            cv2.COLOR_RGB2BGR
        )

        cv2.imwrite(output_path, final_img)

        tmp = processed_counter
        time.sleep(0)
        processed_counter = tmp + 1
        
        current = error_records
        time.sleep(0)
        error_records = current + [{
            "image": img_path,
            "status": "success"
        }]
        
        print("Processed:", img_path)

        return 1

    except Exception as e:

        current = error_records
        time.sleep(0)
        error_records = current + [{
            "image": img_path,
            "status": str(e)
        }]

        print("Error:", img_path, e)

    return 0

# PARALLEL PROCESSING

def parallel_preprocess():

    global processed_counter,error_records

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(EDGES_DIR, exist_ok=True)

    tasks = []

    # Prepare image tasks
    for class_name in os.listdir(INPUT_DIR):

        class_path = os.path.join(INPUT_DIR, class_name)

        if not os.path.isdir(class_path):
            continue

        output_class = os.path.join(
            OUTPUT_DIR, class_name
        )

        edges_class = os.path.join(
            EDGES_DIR, class_name
        )

        os.makedirs(output_class, exist_ok=True)
        os.makedirs(edges_class, exist_ok=True)

        for img_name in os.listdir(class_path):

            img_path = os.path.join(
                class_path, img_name
            )

            output_path = os.path.join(
                output_class, img_name
            )

            edges_path = os.path.join(
                edges_class, img_name
            )

            tasks.append(
                (img_path, output_path, edges_path)
            )

    # Reset counter and records
    processed_counter = 0
    error_records = []

    start_time = time.time()

    # Execute tasks in parallel
    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        results = list(
            executor.map(process_image, tasks)
        )

    end_time = time.time()

    # REDUCTION
    

    total_processed = sum(results)

    total_failed = len(results) - total_processed

    print("\n========== RESULTS ==========")

    print("Total images:", len(tasks))

    print("Successfully processed:", total_processed)

    print("Shared counter", processed_counter)

    print("Shared records:", len(error_records))

    print("Parallel Processing Time:",
          round(end_time - start_time, 2), "seconds")

    print("Workers:", MAX_WORKERS)

    print("=============================")


if __name__ == "__main__":

    print("OpenCV threads:", cv2.getNumThreads())

    parallel_preprocess()



OpenCV threads: 1
Processed: images_dataset\AesleaS10\test_s10.o10.4.png
Processed: images_dataset\AesleaS10\test_s10.o10.6.png
Processed: images_dataset\AesleaS10\test_s10.orr10.3.png
Processed: images_dataset\AesleaS10\test_s10.s2.6.png
Processed: images_dataset\AesleaS10\test_s10.s2.7b.png
Processed: images_dataset\AesleaS10\test_s10.s4.19b.png
Processed: images_dataset\AesleaS10\test_s10.s3.14b.png
Processed: images_dataset\AesleaS10\test_s10.s4.17f.png
Processed: images_dataset\AesleaS10\test_s10.s5.22f.png
Processed: images_dataset\AesleaS10\train_s10.o10.7.png
Processed: images_dataset\AesleaS10\train_s10.o10.1.png
Processed: images_dataset\AesleaS10\train_s10.o10.2.png
Processed: images_dataset\AesleaS10\train_s10.o10.10.png
Processed: images_dataset\AesleaS10\train_s10.orr10.1.png
Processed: images_dataset\AesleaS10\train_s10.o10.9.png
Processed: images_dataset\AesleaS10\train_s10.o10.5.png
Processed: images_dataset\AesleaS10\train_s10.orr10.2.png
Processed: images_dataset\Aes